# Measure before optimizing — Python demonstrations
Compilation, reliable timings, and whole-program speedup.

Use the student loop activity before running these demonstrations. Compare implementations
on the same machine. Record the first-call time, warmed-up timings and measurement protocol.
Bytecode inspection is optional and appears after the main path.

In [ ]:
import timeit

def best_time(fn, repeat=7):
    """Time per call (s), %timeit-style: auto-scale then min over `repeat` trials."""
    number = 1
    while timeit.timeit(fn, number=number) < 0.05:
        number *= 10
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number

def ms(t): return f"{t*1e3:.3f} ms"
def us(t): return f"{t*1e6:.2f} µs"


In [ ]:
import numpy as np
N = 1_000_000
v = np.random.rand(N)        # numpy array (contiguous, typed)
v_list = v.tolist()          # the SAME data as a pure Python list
len(v_list), type(v_list[0])

## 1. Compare implementations
Sum the same one million Float64 values. Predict before timing.

In [ ]:
def mysum(x):
    s = 0.0
    for xi in x:
        s += xi
    return s

# ⚠️ PURE Python loop over a list: this is the slow case
t_loop = best_time(lambda: mysum(v_list))
print("pure Python loop :", ms(t_loop))

In [ ]:
# numpy: the loop runs in compiled C, not in Python
t_np = best_time(lambda: np.sum(v))
print("numpy.sum        :", ms(t_np))
print(f"\nnumpy is ~{t_loop/t_np:.0f}× faster than the pure Python loop")

NumPy executes its reduction in compiled library code; a Python loop over the elements still executes in the interpreter. Use the measured speedup from your run.

### Compile a numeric loop with Numba

`numba.njit` compiles certain numeric functions to native code — exactly Julia's mechanism,
added *on top of* Python for a subset of the language.

In [ ]:
try:
    from numba import njit
except ImportError:
    !pip -q install numba
    from numba import njit

@njit
def sum_numba(x):
    s = 0.0
    for i in range(x.shape[0]):
        s += x[i]
    return s

import time
start = time.perf_counter()
first_result = sum_numba(v)           # first call; no earlier call to this specialization
first_seconds = time.perf_counter() - start
assert np.isclose(first_result, np.sum(v))
print("Numba first call:", ms(first_seconds))
t_nb = best_time(lambda: sum_numba(v))
print("numba njit loop  :", ms(t_nb))
print(f"warmed-up speedup vs Python loop: {t_loop/t_nb:.0f}×")

NumPy provides compiled operations. Numba compiles supported numeric functions for their argument types; Julia also compiles user-written loops. Separate first-call from warmed-up time.

## 2. First call and subsequent calls
Time each call on one million Float64 values. A fresh Numba function makes its first-call
cost observable. Compiled specializations, library initialization and CPU/cache warmup can
all affect early calls. Report the first call separately; do not assume every initial spike
is exclusively compilation. The plot uses elapsed time per call.

In [ ]:
import time
import matplotlib.pyplot as plt

# dedicated array (10^6) so the pure loop stays practical over many calls
w = np.random.rand(1_000_000)
w_list = w.tolist()

@njit
def sum_njit_warm(x):        # FRESH function: its 1st call triggers compilation
    s = 0.0
    for i in range(x.shape[0]):
        s += x[i]
    return s

def per_call_times(fn, arg, reps):
    """Time (s) of each successive call of fn(arg)."""
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter_ns()
        fn(arg)
        ts.append((time.perf_counter_ns() - t0) * 1e-9)
    return ts

R, R_pure = 400, 40
t_numba = per_call_times(sum_njit_warm, w, R)         # spike on 1st call (compilation)
t_numpy = per_call_times(np.sum, w, R)                # flat (compiled C)
t_pure  = per_call_times(mysum, w_list, R_pure)    # flat but slow (interpreted)

print(f"numba : 1st call = {t_numba[0]*1e3:8.2f} ms | warmed up = {np.median(t_numba[10:])*1e3:.3f} ms")
print(f"numpy : 1st call = {t_numpy[0]*1e3:8.3f} ms | warmed up = {np.median(t_numpy[10:])*1e3:.3f} ms")
print(f"pure  : 1st call = {t_pure[0]*1e3:8.2f} ms | warmed up = {np.median(t_pure[5:])*1e3:.2f} ms")

plt.figure(figsize=(8, 4.5))
plt.plot([t*1e3 for t in t_numba], ".", ms=3, label="numba njit")
plt.plot([t*1e3 for t in t_numpy], ".", ms=3, label="numpy")
plt.plot([t*1e3 for t in t_pure],  ".", ms=3, label="pure python")
plt.yscale("log")
plt.xlabel("iteration (call #)"); plt.ylabel("time / call (ms, log scale)")
plt.title("Warmup: decay of the sum time")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Record the measurement protocol
- **Correctness:** check and tolerance.
- **Input:** size and element type.
- **Environment:** hardware, software versions, workers/threads.
- **Protocol:** warmup, repetitions and reported statistic.
- **Timed scope:** included and excluded costs.

`best_time` reports a minimum across repeated trials, after warmup. A median instead
summarizes a typical observation. State which one you report.

In [ ]:
nbytes = v.nbytes
print(f"numpy.sum : {ms(t_np)}  |  throughput ≈ {nbytes/t_np/1e9:.1f} GB/s")
# %timeit np.sum(v)   # <- the standard tool in interactive Colab

## 4. Measure and check Monte-Carlo π
Time warmed-up calls. Check counting logic on fixed points with a known answer; assess the
random estimator separately using its standard error, approximately 1.64 / sqrt(n).
A statistical tolerance does not replace a code check.

In [ ]:
import random
def pi_python(n):
    hits = 0
    for _ in range(n):
        x = random.random(); y = random.random()
        if x*x + y*y <= 1.0:
            hits += 1
    return 4*hits/n

def pi_numpy(n):
    x = np.random.rand(n); y = np.random.rand(n)
    return 4*np.count_nonzero(x*x + y*y <= 1.0)/n

@njit
def pi_numba(n):
    hits = 0
    for _ in range(n):
        x = np.random.random(); y = np.random.random()
        if x*x + y*y <= 1.0:
            hits += 1
    return 4*hits/n

n = 5_000_000
pi_numba(10)  # warm-up
print("π python :", ms(best_time(lambda: pi_python(n))))
print("π numpy  :", ms(best_time(lambda: pi_numpy(n))))
print("π numba  :", ms(best_time(lambda: pi_numba(n))))

In [ ]:
# Check the counting logic on fixed points: 3 of these 4 points are in the quarter disk
xs = np.array([0.0, 0.5, 1.0, 1.0]); ys = np.array([0.0, 0.5, 0.0, 1.0])
assert np.count_nonzero(xs*xs + ys*ys <= 1.0) == 3

# Check the estimator: error divided by its standard error
p = np.pi / 4
for m in (10**4, 10**5, 10**6):
    est = pi_numpy(m)
    se = 4 * np.sqrt(p * (1 - p) / m)
    print(f"n={m}  estimate={est:.5f}  error/SE={(est - np.pi)/se:.2f}")

## Four rules for reliable comparisons
1. Distinguish first-call and warmed-up times.
2. Record the measurement protocol.
3. Check the code and the statistical result.
4. Measure whole-program speedup, not only the block.

Continue with the separate diagnostic activity. Optional bytecode inspection follows.

## Optional — CPython bytecode

The Python analogue of Julia's `@code_lowered` / `@code_native` is `dis`: we see the **bytecode**
that the CPython virtual machine runs in its interpretation loop — never native code.

In [ ]:
import dis
def f(x):
    return 2*x + 1
dis.dis(f)

`BINARY_OP` looks like a **generic** call: CPython would have to *rediscover* the operand types at
run time and find the right method (`__mul__`, `__add__`…) on **every** operation, on **every** loop
iteration.

Since **3.11**
([PEP 659](https://peps.python.org/pep-0659/)), CPython has a *specializing adaptive interpreter*.
It watches the types actually flowing through each bytecode and **rewrites the instruction in place**
into a type-specialized variant. `dis.dis()` hides this: it defaults to `adaptive=False`, so it shows
you the code *before* the interpreter warmed up.

In [ ]:
def hot_loop(v):
    s = 0.0
    for x in v:
        s += x
    return s

data = [1.0] * 1000
for _ in range(100):                 # warm up: let the interpreter observe the types
    hot_loop(data)

for ins in dis.get_instructions(hot_loop, adaptive=True):
    print(f"  {ins.opname:<24} {ins.argrepr}")

Look at what came back: `FOR_ITER_LIST` and **`BINARY_OP_ADD_FLOAT`**. CPython noticed the values
were floats and specialized the bytecode to them. There is no `__add__` lookup left in this loop.

CPython does specialize, but differently from Julia:

| | CPython 3.11+ | Julia |
|---|---|---|
| unit of specialization | **one bytecode** at a time | the **whole function** |
| when | after warm-up, in place | on the 1st call with those types |
| the type check | a **guard on every execution** (wrong type → deoptimize) | **eliminated** — proved once |
| what runs | still the **interpreter loop** | **machine code** |
| the data | still **boxed** `PyObject*` on the heap | unboxed `Float64` in a register |

`BINARY_OP_ADD_FLOAT` still dispatches through the eval loop, still checks that both operands really
are floats, still unboxes two heap objects to add them, and still boxes the result. It removed the
method lookup — the cheapest part. Julia removes the interpreter, the guard **and** the boxing.

CPython specializes one instruction at a time, keeps the guard, stays in the interpreter and keeps
boxed values. Julia specializes the whole function, drops the guard, emits native code and unboxes.
This is why the gap measured above between the Python loop and the compiled loop remains after
PEP 659.

Julia, for comparison: `f(3.0)` and `f(3)` compile to different native code, and the type check
**disappears** from the hot path. We'll meet this distinction again for **dispatch**, **threads**
and the **GPU**.